# Saving module configurations & re-creating pipeline from config file

### Saving interactively added module configurations to a BioPAP config file

In [1]:
# Make sure you have the most recent version installed.
#%pip install -e ..

In [2]:
from biopap.modules.anonymize import Anonymize
from biopap.modules.artifact_reject import ArtifactReject
from biopap.modules.baseline_correction import BaselineCorrection
from biopap.modules.bids import BIDSConversion
from biopap.modules.blink_detection import BlinkDetection
from biopap.modules.downsample import Downsample
from biopap.modules.eog_channels import EOGChannels
from biopap.modules.iir_filter import IIRFilter
from biopap.modules.load_data import LoadData
from biopap.modules.plot import Plot, PlotTypes
from biopap.modules.rereference import Rereference
from biopap.modules.segmentation import Segmentation
from biopap.modules.save_output import SaveOutput
from biopap.modules.set_channel_type_and_montage import SetChannelTypeAndMontage

from biopap.pipeline import Pipeline
from biopap.modules.helpers import save_modules_config

In [3]:
# Have a data file and load it
data_file = '/data/BioPAP/data/CBN1BD071212-Cropped.bdf'

load = LoadData(file_path=data_file)

# Add the modules you want to use, adding and testing the configuration for
# each until you are satisfied that each module's settings are correct.

ano = Anonymize(
    raw=load.raw,
    daysback=100,
    experimenter='J. Jansen',
    participant_id='AB00123')

Extracting EDF parameters from /data/BioPAP/data/CBN1BD071212-Cropped.bdf...
BDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 3276799  =      0.000 ...  1600.000 secs...


In [4]:
# You can use both the standard variables - in this case `ref_channels` -, as well as additional
# parameters to be passed on the the underlying mne-function, like `ch_type` in the example below.

ref = Rereference(
    raw=ano.processed,
    ref_channels=['EXG1', 'EXG2'],
    ch_type='auto')

EEG channel type selected for re-referencing
Applying a custom ('EEG',) reference.


In [5]:
plt = Plot(
    raw=ano.processed,
    save_folder='./',
    save_postfix='IRR')

eog = EOGChannels(
    raw=ref.processed,
    ref_channels=['hEOG', 'vEOG'],
    anode_channels=['EXG3','EXG5'],
    cathode_channels=['EXG4','EXG6'])

channel_types = {
    'misc': ['EXG1','EXG2','EXG3','EXG4','EXG5','EXG6','EXG7','EXG8','Erg1','Erg2','Plet'],
    'gsr': ['GSR1', 'GSR2'],
    'resp': ['Resp'],
    'temperature': ['Temp']
}

montage = SetChannelTypeAndMontage(
    raw=eog.processed,
    channel_types=channel_types,
    mne_standard_montage='biosemi64'
)

flt = IIRFilter(
    raw=montage.processed,
    l_freq=0.1,
    h_freq=35)

dwn = Downsample(
    raw=flt.processed,
    sfreq=500)  # Downsample to 500 Hz

blink = BlinkDetection(raw=dwn.processed)

Finding events on: Status
Trigger channel Status has a non-zero initial value of 65536 (consider using initial_event=True to detect this event)
Removing orphaned offset at the beginning of the file.
2042 events found on stim channel Status
Event IDs: [  1   2   3   4   5   6   9  10  11  12  13  14  15  16  17  18  19  20
  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35  36  37  38
  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53  54  55  56
  57  58  59  60  61  62  63  64  65  66  67  68  69  96  97  98  99 101
 102 103 104 105 106 121 122 123 124 125 126 141 142 143 144 145 146 161
 162 163 164 165 201 203 211 212 213 231 232 233 240 241 253]


2026-09-29 13:20:43,447:INFO:acceleratesupport:No OpenGL_accelerate module loaded: No module named 'OpenGL_accelerate'


Using qt as 2D backend.
wrote plot to 'AB00123_plot_IRR_(2).png'
EEG channel type selected for re-referencing
Creating RawArray with float64 data, n_channels=2, n_times=3276800
    Range : 0 ... 3276799 =      0.000 ...  1600.000 secs
Ready.
Added the following bipolar channels:
hEOG, vEOG
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.1 - 35 Hz

IIR filter parameters
---------------------
Butterworth bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 16 (effective, after forward-backward)
- Cutoffs at 0.10, 35.00 Hz: -6.02, -6.02 dB



/home/maarten/Documents/Projects/BioPAP/BioPAP/src/biopap/modules/set_channel_type_and_montage.py:71: RuntimeWarning: The unit for channel(s) EXG1, EXG2, EXG3, EXG4, EXG5, EXG6, EXG7, EXG8, Erg1, Erg2, Plet has changed from V to NA.
  self.processed.set_channel_types({chs: type for chs in channel_types.get(type)}, **s_kwargs)
/home/maarten/Documents/Projects/BioPAP/BioPAP/src/biopap/modules/set_channel_type_and_montage.py:71: RuntimeWarning: The unit for channel(s) GSR1, GSR2 has changed from V to S.
  self.processed.set_channel_types({chs: type for chs in channel_types.get(type)}, **s_kwargs)
/home/maarten/Documents/Projects/BioPAP/BioPAP/src/biopap/modules/set_channel_type_and_montage.py:71: RuntimeWarning: The unit for channel(s) Temp has changed from V to C.
  self.processed.set_channel_types({chs: type for chs in channel_types.get(type)}, **s_kwargs)


Finding events on: Status
Trigger channel Status has a non-zero initial value of 65536 (consider using initial_event=True to detect this event)
Removing orphaned offset at the beginning of the file.
2042 events found on stim channel Status
Event IDs: [  1   2   3   4   5   6   9  10  11  12  13  14  15  16  17  18  19  20
  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35  36  37  38
  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53  54  55  56
  57  58  59  60  61  62  63  64  65  66  67  68  69  96  97  98  99 101
 102 103 104 105 106 121 122 123 124 125 126 141 142 143 144 145 146 161
 162 163 164 165 201 203 211 212 213 231 232 233 240 241 253]
Finding events on: Status
Trigger channel Status has a non-zero initial value of 65536 (consider using initial_event=True to detect this event)
Removing orphaned offset at the beginning of the file.
2041 events found on stim channel Status
Event IDs: [  1   2   3   4   5   6   9  10  11  12  13  14  15  16  17  18  19  2

/home/maarten/Documents/Projects/BioPAP/BioPAP/src/biopap/modules/downsample.py:53: RuntimeWarning: Resampling of the stim channels caused event information to become unreliable. Consider finding events on the original data and passing the event matrix as a parameter.
  self.processed.resample(** {


In [6]:
# Make sure when you have multiple instances of the same module
# in your pipeline to give each instance a different name (`plt` vs `plt2`).

plt2 = Plot(
    raw=blink.processed,
    title="Show blink annotations")

Finding events on: Status
Trigger channel Status has a non-zero initial value of 65536 (consider using initial_event=True to detect this event)
Removing orphaned offset at the beginning of the file.
2041 events found on stim channel Status
Event IDs: [  1   2   3   4   5   6   9  10  11  12  13  14  15  16  17  18  19  20
  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35  36  37  38
  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53  54  55  56
  57  58  59  60  61  62  63  64  65  66  67  68  69  96  97  98  99 101
 102 103 104 105 106 121 122 123 124 125 126 141 142 143 144 145 146 161
 162 163 164 165 201 203 211 212 213 231 232 233 240 241 253]
Channels marked as bad:
none


In [7]:
seg = Segmentation(
    raw=blink.processed,  
    stim_channel='Status',  
    start=-0.2,  
    end=0.8,     
    baseline=(None, 0),
    reference_markers={'range': '1-69'})

artifact = ArtifactReject(
    raw=seg.processed,
    mode='interpolation',
    max_gradient_uv=50,
    max_amplitude_uv=75,
    min_activity_uv=0.5,
    interval_ms=100)

baseline = BaselineCorrection(
    raw=artifact.processed,
    baseline=(-0.2, 0))

pre_ica_save = SaveOutput(
    raw=baseline.processed,
    export_formats='fif',
    output_file='./partially_processed.fif')

plt3 = Plot(
    raw=pre_ica_save.processed,
    title="Preprocessed data")

Finding events on: Status
Trigger channel Status has a non-zero initial value of 65536 (consider using initial_event=True to detect this event)
Removing orphaned offset at the beginning of the file.
2041 events found on stim channel Status
Event IDs: [  1   2   3   4   5   6   9  10  11  12  13  14  15  16  17  18  19  20
  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35  36  37  38
  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53  54  55  56
  57  58  59  60  61  62  63  64  65  66  67  68  69  96  97  98  99 101
 102 103 104 105 106 121 122 123 124 125 126 141 142 143 144 145 146 161
 162 163 164 165 201 203 211 212 213 231 232 233 240 241 253]
Not setting metadata
540 matching events found
Setting baseline interval to [-0.2, 0.0] s
Applying baseline correction (mode: mean)
0 projection items activated
Using data from preloaded Raw for 540 events and 501 original time points ...
20 bad epochs dropped
Fp1: bad in 3 epochs
AF7: bad in 2 epochs
AF3: bad in 1 epoch

/home/maarten/Documents/Projects/BioPAP/BioPAP/src/biopap/modules/save_output.py:108: RuntimeWarning: This filename (partially_processed.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  self.processed.save(f_new, overwrite=overwrite)


wrote 'partially_processed.fif' (85,479,520)
You seem to have overlapping epochs. Some event lines may be duplicated in the plot.


/home/maarten/Documents/Projects/BioPAP/BioPAP/src/biopap/modules/plot.py:177: RuntimeWarning: More events than default colors available. You should pass a list of unique colors.
  _ = self.processed.plot(** args | { 'block': True, 'show': True } | kwargs)


Dropped 0 epochs: 
The following epochs were marked as bad and are dropped:
[]
Channels marked as bad:
none


In [8]:
save_output = SaveOutput(
    raw=pre_ica_save.processed,
    export_formats='fif',
    output_file='./processed_data-epo.fif',
    overwrite=True)

Overwriting existing file.
Overwriting existing file.
Overwriting existing file.
wrote 'processed_data-epo.fif' (85,479,520)


Once you are done with adding modules and editing their configuration, the settings for the complete pipeline can be saved to a configuration file by calling the `save_modules_config` helper function.

The order of the modules is important, as this will also be the order in which the modules will be added when a new pipeline is created from the saved configuration file.

Note that you only need to pass the function a list of the modules that you actually want added to the config. For instance, the example below omits `plt2`, the second plotting step.

Also note that it is not necessary to save the settings from the `Load` module, as that module is also integrated in the `Pipeline` class.

In [9]:
cfg_file = './config.yaml'
save_modules_config(
    modules=[ano, ref, plt, eog, flt, dwn, seg, artifact, baseline, pre_ica_save, plt3, save_output], 
    config_path=cfg_file
)

Please note: besides saving a pipeline configuration from individual modules as shown above, you can also first manually create the pipeline, add all the steps and their configuration, and call the pipeline's `to_config()` method.

# Re-creating pipeline from config file

By loading a saved config file into a new `Pipeline`-instance, you automatically create the complete pipeline with all the specified settings.

In [10]:
cfg_file = './config.yaml'

new_pipe = Pipeline.from_config(
    config=cfg_file,
    input_file=data_file,
    subject_id='ABC123'
)

new_pipe.inspect()

2026-09-29 13:21:58,461:INFO:pipeline:LoadData:Extracting EDF parameters from /data/BioPAP/data/CBN1BD071212-Cropped.bdf...
2026-09-29 13:21:58,462:INFO:pipeline:LoadData:BDF file detected
2026-09-29 13:21:58,462:INFO:pipeline:LoadData:Setting channel info structure...
2026-09-29 13:21:58,462:INFO:pipeline:LoadData:Creating raw.info structure...
2026-09-29 13:21:58,463:INFO:pipeline:LoadData:Reading 0 ... 3276799  =      0.000 ...  1600.000 secs...
2026-09-29 13:21:58,463:INFO:pipeline:Loaded '/data/BioPAP/data/CBN1BD071212-Cropped.bdf'


00. Anonymize
    daysback: 100
    experimenter: 'J. Jansen'
    participant_id: 'AB00123'

01. Rereference *
    ch_type: 'auto'
    ref_channels: ['EXG1', 'EXG2']

02. Plot
    plot_types: 1
    save_folder: './'
    save_overwrite: False
    save_postfix: 'IRR'

03. EOGChannels *
    anode_channels: ['EXG3', 'EXG5']
    cathode_channels: ['EXG4', 'EXG6']
    drop_refs: False
    ref_channels: ['hEOG', 'vEOG']

04. IIRFilter *
    ftype: 'butter'
    h_freq: 35
    l_freq: 0.1
    method: 'iir'
    order: 4

05. Downsample *
    resample_events: True
    sfreq: 500

06. Segmentation *
    baseline: [None, 0]
    end: 0.8
    reference_markers:
      range: '1-69'
    start: -0.2
    stim_channel: 'Status'

07. ArtifactReject *
    interval_ms: 100
    max_amplitude_uv: 75
    max_gradient_uv: 50
    min_activity_uv: 0.5
    mode: 'interpolation'

08. BaselineCorrection *
    baseline: [-0.2, 0]

09. SaveOutput
    _intermediate: True
    _steps_completed: []
    export_formats: 'fif

## A word on subject_id

The subject ID can be used to differentiate between outputs for different users when running the pipeline in batches. When you use the string `{subject_id}` in the value of any string-type config variable, it will be automatically replaced by the value of subject_id. So, for instance, by defining a pipeline-step thus:

```python
SaveOutput(
    raw=baseline.processed,
    export_formats='fif',
    output_file='./partially_processed_{subject_id}.fif',
    overwrite=True)
```
running the pipeline with `subject_id="A123"` will result in an outputfile `partially_processed_A123.fif`.

If you do not specify a subject_id when creating the pipeline, the program will use the input file's name without the suffix. It will first try to read this from the 'filenames' attribute from the file's metadata, and if that is empty, use the actual file's path.